### **Rahven's VENTURI**

#### Notebook for preparing FastF1 datset

In [1]:
from pathlib import Path
import json
import pandas as pd

In [2]:

# Project Paths

PROJECT_ROOT = Path(r"C:\F1-AI")

RAW_FASTF1_DIR = PROJECT_ROOT / "data" / "raw" / "fastf1"

assert RAW_FASTF1_DIR.exists(), (
    f"FastF1 directory not found:\n{RAW_FASTF1_DIR}"
)

print(f"FastF1 Root : {RAW_FASTF1_DIR}")

FastF1 Root : C:\F1-AI\data\raw\fastf1


In [3]:

# Discover all parquet files

parquet_files = sorted(
    RAW_FASTF1_DIR.rglob("*.parquet")
)

print(f"Total parquet files : {len(parquet_files):,}")

Total parquet files : 36,569


In [4]:

# Discover Logical Dataset Types 

from collections import Counter

dataset_counter = Counter()

for file in parquet_files:
    relative_parts = file.relative_to(RAW_FASTF1_DIR).parts

    # Driver-specific datasets
    if len(relative_parts) >= 5 and relative_parts[-2] in ("car_data", "position_data"):
        dataset_name = relative_parts[-2]

    # Session-level datasets
    else:
        dataset_name = file.stem

    dataset_counter[dataset_name] += 1

dataset_summary = (
    pd.DataFrame.from_dict(
        dataset_counter,
        orient="index",
        columns=["files"]
    )
    .sort_values("files", ascending=False)
)

dataset_summary

,files
car_data,17013
position_data,16994
results,857
weather,855
laps,850


In [5]:
# Build FastF1 Dataset Catalog
 

catalog = []

for dataset_name in sorted(dataset_counter.keys()):

    # Files belonging to this logical dataset
    if dataset_name in ("car_data", "position_data"):
        files = [
            f for f in parquet_files
            if f.parent.name == dataset_name
        ]
        driver_specific = True

    else:
        files = [
            f for f in parquet_files
            if f.stem == dataset_name
        ]
        driver_specific = False

    sample_file = files[0].relative_to(RAW_FASTF1_DIR)

    catalog.append({
        "dataset": dataset_name,
        "files": len(files),
        "driver_specific": driver_specific,
        "sample_file": str(sample_file)
    })

dataset_catalog = (
    pd.DataFrame(catalog)
      .sort_values("files", ascending=False)
      .reset_index(drop=True)
)

dataset_catalog

,dataset,files,driver_specific,sample_file
0,car_data,17013,True,2018\Abu_Dhabi_Grand_Prix\Practice_1\car_data\...
1,position_data,16994,True,2018\Abu_Dhabi_Grand_Prix\Practice_1\position_...
2,results,857,False,2018\Abu_Dhabi_Grand_Prix\Practice_1\results.p...
3,weather,855,False,2018\Abu_Dhabi_Grand_Prix\Practice_1\weather.p...
4,laps,850,False,2018\Abu_Dhabi_Grand_Prix\Practice_1\laps.parquet


In [6]:

# Representative File for Each Dataset


representative_files = {}

for file in parquet_files:
    relative_parts = file.relative_to(RAW_FASTF1_DIR).parts

    if len(relative_parts) >= 5 and relative_parts[-2] in ("car_data", "position_data"):
        dataset = relative_parts[-2]
    else:
        dataset = file.stem

    if dataset not in representative_files:
        representative_files[dataset] = file

for dataset, file in representative_files.items():
    print(f"{dataset:15} -> {file.relative_to(RAW_FASTF1_DIR)}")

car_data        -> 2018\Abu_Dhabi_Grand_Prix\Practice_1\car_data\ALO.parquet
laps            -> 2018\Abu_Dhabi_Grand_Prix\Practice_1\laps.parquet
position_data   -> 2018\Abu_Dhabi_Grand_Prix\Practice_1\position_data\ALO.parquet
results         -> 2018\Abu_Dhabi_Grand_Prix\Practice_1\results.parquet
weather         -> 2018\Abu_Dhabi_Grand_Prix\Practice_1\weather.parquet


In [7]:
# Schema Inspection Utility

def inspect_schema(file_path: Path):
    """
    Inspect a representative parquet file and
    return dataset metadata.
    """

    df = pd.read_parquet(file_path)

    return {
        "rows": len(df),
        "columns": len(df.columns),
        "memory_mb": round(
            df.memory_usage(deep=True).sum() / 1024**2,
            2
        ),
        "column_names": list(df.columns),
        "dtypes": df.dtypes.to_dict()
    }

In [8]:
# Inspect Representative Files

schema_catalog = {}

for dataset, file in representative_files.items():
    schema_catalog[dataset] = inspect_schema(file)

In [9]:
# Schema Summary

summary = []

for dataset, info in schema_catalog.items():

    summary.append({
        "dataset": dataset,
        "rows": info["rows"],
        "columns": info["columns"],
        "memory_mb": round(info["memory_mb"], 2)
    })

schema_summary = (
    pd.DataFrame(summary)
      .sort_values("dataset")
      .reset_index(drop=True)
)

schema_summary

,dataset,rows,columns,memory_mb
0,car_data,27177,10,3.03
1,laps,460,31,0.21
2,position_data,22272,8,3.31
3,results,20,22,0.02
4,weather,111,8,0.01


In [10]:
# Build Master Schema Catalog

schema_rows = []

for dataset, metadata in schema_catalog.items():

    for column in metadata["column_names"]:

        schema_rows.append({
            "dataset": dataset,
            "column": column,
            "dtype": str(metadata["dtypes"][column])
        })

master_schema = (
    pd.DataFrame(schema_rows)
      .sort_values(["dataset", "column"])
      .reset_index(drop=True)
)

master_schema

,dataset,column,dtype
0,car_data,Brake,bool
1,car_data,DRS,int64
2,car_data,Date,datetime64[ns]
3,car_data,RPM,float64
4,car_data,SessionTime,timedelta64[ns]
...,...,...,...
74,weather,Rainfall,bool
75,weather,Time,timedelta64[ns]
76,weather,TrackTemp,float64
77,weather,WindDirection,int64


In [11]:
# Dataset Profiling Utility

def profile_dataset(file_path: Path):
    """
    Generate a basic profile for a representative dataset.
    """

    df = pd.read_parquet(file_path)

    profile = pd.DataFrame({
        "column": df.columns,
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum().values,
        "missing_pct": (
            df.isna().mean().values * 100
        ).round(2),
        "unique": df.nunique(dropna=True).values
    })

    return df, profile

In [12]:
# Dataset Report

def dataset_report(dataset_name: str):
    """
    Generate a standardized report for a representative dataset.
    """

    file_path = representative_files[dataset_name]
    df = pd.read_parquet(file_path)

    
    print(f"DATASET : {dataset_name.upper()}")
    

    print(f"Representative File : {file_path.relative_to(RAW_FASTF1_DIR)}")
    print(f"Rows                : {len(df):,}")
    print(f"Columns             : {len(df.columns)}")
    print(
        f"Memory              : "
        f"{df.memory_usage(deep=True).sum()/1024**2:.2f} MB"
    )

    profile = pd.DataFrame({
        "Column": df.columns,
        "Dtype": df.dtypes.astype(str),
        "Missing": df.isna().sum().values,
        "Missing %": (df.isna().mean() * 100).round(2).values,
        "Unique": df.nunique(dropna=True).values
    })

    return profile

-------------

### **Engineering Notes – laps**

**Granularity**
- One row represents one completed lap by one driver.

**Purpose**
- Central dataset for race/session analysis.

**Potential Join Keys**
- Driver
- DriverNumber
- LapNumber
- Team
- Time

**Contains**
- Lap timing
- Tyre information
- Sector times
- Speed traps
- Track status

**Remarks**
- Likely to become the central fact table during dataset integration.

In [13]:
dataset_report("laps")

DATASET : LAPS
Representative File : 2018\Abu_Dhabi_Grand_Prix\Practice_1\laps.parquet
Rows                : 460
Columns             : 31
Memory              : 0.21 MB


,Column,Dtype,Missing,Missing %,Unique
Time,Time,timedelta64[ns],0,0.00,460
Driver,Driver,object,0,0.00,20
DriverNumber,DriverNumber,object,0,0.00,20
LapTime,LapTime,timedelta64[ns],148,32.17,309
LapNumber,LapNumber,float64,0,0.00,29
Stint,Stint,float64,0,0.00,7
PitOutTime,PitOutTime,timedelta64[ns],359,78.04,101
PitInTime,PitInTime,timedelta64[ns],359,78.04,101
Sector1Time,Sector1Time,timedelta64[ns],87,18.91,358
Sector2Time,Sector2Time,timedelta64[ns],2,0.43,450


----------

### **Dataset : results**

#### Purpose
Official summary of a session.

Each row corresponds to one participating driver.

#### Granularity
- One row = One Driver × One Session

#### Contains
- Driver identity
- Team information
- Session classification
- Official timing (when applicable)
- Grid position (qualifying/race)
- Championship points (race)

#### Expected Size
~20 rows per session.

#### Potential Join Keys
- DriverNumber
- DriverId
- FullName

#### Notes
Useful for attaching driver metadata and official session outcomes to analytical datasets.

In [14]:
dataset_report("results")

DATASET : RESULTS
Representative File : 2018\Abu_Dhabi_Grand_Prix\Practice_1\results.parquet
Rows                : 20
Columns             : 22
Memory              : 0.02 MB


,Column,Dtype,Missing,Missing %,Unique
DriverNumber,DriverNumber,object,0,0.0,20
BroadcastName,BroadcastName,object,0,0.0,20
Abbreviation,Abbreviation,object,0,0.0,20
DriverId,DriverId,object,0,0.0,19
TeamName,TeamName,object,0,0.0,10
TeamColor,TeamColor,object,0,0.0,10
TeamId,TeamId,object,0,0.0,11
FirstName,FirstName,object,0,0.0,20
LastName,LastName,object,0,0.0,20
FullName,FullName,object,0,0.0,20


--------------

### **Dataset : weather**

#### Purpose
Environmental conditions recorded during the session.

#### Granularity
- One row = One Weather Observation

#### Contains
- Air temperature
- Track temperature
- Humidity
- Wind speed
- Wind direction
- Atmospheric pressure
- Rain indicator

#### Expected Size
~100 observations per session.

#### Potential Join Keys
- Time

#### Notes
Weather is a valuable contextual dataset and may later be aligned with laps or telemetry using timestamps.

In [15]:
dataset_report("weather")

DATASET : WEATHER
Representative File : 2018\Abu_Dhabi_Grand_Prix\Practice_1\weather.parquet
Rows                : 111
Columns             : 8
Memory              : 0.01 MB


,Column,Dtype,Missing,Missing %,Unique
Time,Time,timedelta64[ns],0,0.0,111
AirTemp,AirTemp,float64,0,0.0,21
Humidity,Humidity,float64,0,0.0,55
Pressure,Pressure,float64,0,0.0,18
Rainfall,Rainfall,bool,0,0.0,1
TrackTemp,TrackTemp,float64,0,0.0,35
WindDirection,WindDirection,int64,0,0.0,69
WindSpeed,WindSpeed,float64,0,0.0,23


---------------------

### **Dataset : car_data**

#### Purpose
High-frequency telemetry captured from an individual car.

#### Granularity
- One row = One Telemetry Sample

#### Contains
- Speed
- RPM
- Gear
- Throttle
- Brake
- DRS status
- Session timestamps

#### Expected Size
Tens of thousands of rows per driver per session.

#### Potential Join Keys
- Time
- SessionTime

#### Notes
This dataset should **not** be merged directly with laps because of the difference in granularity.

Instead, telemetry should later be aggregated or aligned to laps before integration.

In [16]:
dataset_report("car_data")

DATASET : CAR_DATA
Representative File : 2018\Abu_Dhabi_Grand_Prix\Practice_1\car_data\ALO.parquet
Rows                : 27,177
Columns             : 10
Memory              : 3.03 MB


,Column,Dtype,Missing,Missing %,Unique
Date,Date,datetime64[ns],0,0.0,27177
RPM,RPM,float64,0,0.0,4669
Speed,Speed,float64,0,0.0,314
nGear,nGear,int64,0,0.0,9
Throttle,Throttle,float64,0,0.0,100
Brake,Brake,bool,0,0.0,2
DRS,DRS,int64,0,0.0,4
Source,Source,object,0,0.0,1
Time,Time,timedelta64[ns],0,0.0,27177
SessionTime,SessionTime,timedelta64[ns],0,0.0,27177


--------------------

### **Dataset : position_data**

#### Purpose
Track position of a driver throughout the session.

#### Granularity
- One row = One Position Sample

#### Contains
- X, Y, Z coordinates
- Session timestamps
- Position status

#### Expected Size
Tens of thousands of rows per driver per session.

#### Potential Join Keys
- Time
- SessionTime

#### Notes
Useful for track mapping, racing line analysis, overtakes, corner behavior, and visualization.

In [17]:
dataset_report("position_data")

DATASET : POSITION_DATA
Representative File : 2018\Abu_Dhabi_Grand_Prix\Practice_1\position_data\ALO.parquet
Rows                : 22,272
Columns             : 8
Memory              : 3.31 MB


,Column,Dtype,Missing,Missing %,Unique
Date,Date,datetime64[ns],0,0.0,22272
Status,Status,object,0,0.0,1
X,X,float64,0,0.0,2767
Y,Y,float64,0,0.0,3199
Z,Z,float64,0,0.0,105
Source,Source,object,0,0.0,1
Time,Time,timedelta64[ns],0,0.0,22272
SessionTime,SessionTime,timedelta64[ns],0,0.0,22272


____________________________________

------------------------------------


                     SESSION
                        │
        ┌───────────────┼────────────────┐
        │               │                │
        │               │                │
     RESULTS         WEATHER           LAPS
                                         │
                                         │
                                One lap per driver
                                         │
                     ┌───────────────────┴────────────────────┐
                     │                                        │
                 CAR DATA                                POSITION DATA
            (Telemetry samples)                     (GPS position samples)


----------------------------------------

### **First Principle**

**Never merge because we can.**

**Merge because the merged table improves prediction.**

---------------------

In [18]:

# Candidate Key Validator

def validate_candidate_key(df, columns):
    """
    Validate whether a set of columns uniquely identifies rows.
    """

    duplicate_rows = df.duplicated(subset=columns).sum()

    total_rows = len(df)

    unique_rows = df[columns].drop_duplicates().shape[0]

    return {
        "candidate_key": " + ".join(columns),
        "total_rows": total_rows,
        "unique_combinations": unique_rows,
        "duplicate_rows": duplicate_rows,
        "is_valid": duplicate_rows == 0
    }

In [19]:
def show_key_validation(df, dataset_name, candidate_keys):

    results = []

    for key in candidate_keys:

        result = validate_candidate_key(df, key)

        results.append(result)

    display(pd.DataFrame(results))

**LAPS**

In [20]:
laps_df = pd.read_parquet(representative_files["laps"])

show_key_validation(
    laps_df,
    "laps",
    [
        ["DriverNumber"],
        ["LapNumber"],
        ["DriverNumber", "LapNumber"],
        ["Driver", "LapNumber"],
        ["Driver", "Time"]
    ]
)

,candidate_key,total_rows,unique_combinations,duplicate_rows,is_valid
0,DriverNumber,460,20,440,False
1,LapNumber,460,29,431,False
2,DriverNumber + LapNumber,460,460,0,True
3,Driver + LapNumber,460,460,0,True
4,Driver + Time,460,460,0,True


##### Candidate Key Analysis – laps

##### Valid Candidate Keys

| Key | Recommendation |
|------|----------------|
| DriverNumber + LapNumber |  Preferred |
| Driver + LapNumber |  Good Alternative |
| Driver + Time |  Acceptable but not preferred |

### Decision

The preferred logical key for the laps dataset is:

Season
+ GrandPrix
+ Session
+ DriverNumber
+ LapNumber

The first three components are currently encoded in the directory structure and will later be added as explicit metadata columns.

**RESULTS**

In [21]:
results_df = pd.read_parquet(representative_files["results"])

show_key_validation(
    results_df,
    "results",
    [
        ["DriverNumber"],
        ["DriverId"],
        ["FullName"],
        ["DriverNumber", "TeamName"]
    ]
)

,candidate_key,total_rows,unique_combinations,duplicate_rows,is_valid
0,DriverNumber,20,20,0,True
1,DriverId,20,19,1,False
2,FullName,20,20,0,True
3,DriverNumber + TeamName,20,20,0,True


In [22]:
results_df[
    results_df.duplicated("DriverId", keep=False)
][
    ["DriverNumber",
     "DriverId",
     "FullName"]
]

,DriverNumber,DriverId,FullName
15,36,nan,Antonio Giovinazzi
16,40,nan,Robert Kubica


**WEATHER**

In [23]:
weather_df = pd.read_parquet(representative_files["weather"])

show_key_validation(
    weather_df,
    "weather",
    [
        ["Time"]
    ]
)

,candidate_key,total_rows,unique_combinations,duplicate_rows,is_valid
0,Time,111,111,0,True


**CAR DATA**

In [24]:
car_df = pd.read_parquet(representative_files["car_data"])

show_key_validation(
    car_df,
    "car_data",
    [
        ["Time"],
        ["SessionTime"],
        ["Date"]
    ]
)

,candidate_key,total_rows,unique_combinations,duplicate_rows,is_valid
0,Time,27177,27177,0,True
1,SessionTime,27177,27177,0,True
2,Date,27177,27177,0,True


**POSITION DATA**

In [25]:
position_df = pd.read_parquet(representative_files["position_data"])

show_key_validation(
    position_df,
    "position_data",
    [
        ["Time"],
        ["SessionTime"],
        ["Date"]
    ]
)

,candidate_key,total_rows,unique_combinations,duplicate_rows,is_valid
0,Time,22272,22272,0,True
1,SessionTime,22272,22272,0,True
2,Date,22272,22272,0,True


-------------------------------------

> **Important**
>
> Driver-specific datasets (`car_data`, `position_data`) do not contain driver identifiers as columns.
>
> The driver identity is encoded in the filename (e.g., `ALO.parquet`).
>
> During integration, the driver identifier must be extracted from the filename and added as an explicit column before any concatenation or merge operations.

------------------------

### **Archive-wide Validation**

#### Objective

The analyses performed on representative files provide useful hypotheses,
but they are insufficient for designing a robust integration pipeline.

This section validates those hypotheses across the complete FastF1 archive.

The goals are to:

- Verify candidate keys.
- Detect malformed datasets.
- Identify inconsistencies.
- Confirm assumptions before integration.
- Build evidence-based merge strategies.

In [26]:

# Parse File Metadata

def parse_file_metadata(file_path: Path):

    parts = file_path.relative_to(RAW_FASTF1_DIR).parts

    if parts[-2] in ("car_data", "position_data"):

        return {
            "season": int(parts[0]),
            "grand_prix": parts[1],
            "session": parts[2],
            "dataset": parts[-2],
            "driver": file_path.stem
        }

    return {
        "season": int(parts[0]),
        "grand_prix": parts[1],
        "session": parts[2],
        "dataset": file_path.stem,
        "driver": None
    }

In [27]:
parse_file_metadata(parquet_files[0])

{'season': 2018,
 'grand_prix': 'Abu_Dhabi_Grand_Prix',
 'session': 'Practice_1',
 'dataset': 'car_data',
 'driver': 'ALO'}

In [28]:
# Validate Candidate Key

def archive_validate_key(file_path, candidate_key):

    df = pd.read_parquet(file_path)

    missing_columns = [
        c for c in candidate_key
        if c not in df.columns
    ]

    if missing_columns:

        return {
            "status": "missing_columns",
            "duplicates": None
        }

    duplicates = df.duplicated(candidate_key).sum()

    return {
        "status": "passed" if duplicates == 0 else "failed",
        "duplicates": duplicates
    }

In [30]:
# Validate Entire Archive

from collections import defaultdict

validation_plan = {

    "laps": [
        ["DriverNumber", "LapNumber"]
    ],

    "results": [
        ["DriverNumber"]
    ],

    "weather": [
        ["Time"]
    ],

    "car_data": [
        ["Time"],
        ["SessionTime"]
    ],

    "position_data": [
        ["Time"],
        ["SessionTime"]
    ]

}

report = []

for file in parquet_files:

    meta = parse_file_metadata(file)

    dataset = meta["dataset"]

    if dataset not in validation_plan:
        continue

    for key in validation_plan[dataset]:

        result = archive_validate_key(file, key)

        report.append({

            "dataset": dataset,

            "season": meta["season"],

            "grand_prix": meta["grand_prix"],

            "session": meta["session"],

            "driver": meta["driver"],

            "candidate_key": " + ".join(key),

            "status": result["status"],

            "duplicates": result["duplicates"]

        })

validation_report = pd.DataFrame(report)

validation_report.head()

,dataset,season,grand_prix,session,driver,candidate_key,status,duplicates
0,car_data,2018,Abu_Dhabi_Grand_Prix,Practice_1,ALO,Time,passed,0
1,car_data,2018,Abu_Dhabi_Grand_Prix,Practice_1,ALO,SessionTime,passed,0
2,car_data,2018,Abu_Dhabi_Grand_Prix,Practice_1,BOT,Time,passed,0
3,car_data,2018,Abu_Dhabi_Grand_Prix,Practice_1,BOT,SessionTime,passed,0
4,car_data,2018,Abu_Dhabi_Grand_Prix,Practice_1,ERI,Time,passed,0


In [31]:
summary = (

    validation_report

    .groupby(

        ["dataset",
         "candidate_key",
         "status"]

    )

    .size()

    .unstack(fill_value=0)

    .reset_index()

)

summary

status,dataset,candidate_key,failed,passed
0,car_data,SessionTime,0,17013
1,car_data,Time,0,17013
2,laps,DriverNumber + LapNumber,0,850
3,position_data,SessionTime,0,16994
4,position_data,Time,0,16994
5,results,DriverNumber,0,857
6,weather,Time,6,849


In [32]:
weather_failures = validation_report[
    (validation_report["dataset"] == "weather") &
    (validation_report["status"] == "failed")
]

weather_failures

,dataset,season,grand_prix,session,driver,candidate_key,status,duplicates
17778,weather,2020,Austrian_Grand_Prix,Practice_1,None,Time,failed,370
56635,weather,2024,Las_Vegas_Grand_Prix,Practice_2,None,Time,failed,46
69002,weather,2025,Singapore_Grand_Prix,Practice_1,None,Time,failed,11
69085,weather,2025,Singapore_Grand_Prix,Practice_2,None,Time,failed,73
69168,weather,2025,Singapore_Grand_Prix,Practice_3,None,Time,failed,8
69583,weather,2025,Spanish_Grand_Prix,Practice_3,None,Time,failed,5


In [33]:
failed = weather_failures.iloc[0]

weather_path = (
    RAW_FASTF1_DIR
    / str(failed["season"])
    / failed["grand_prix"]
    / failed["session"]
    / "weather.parquet"
)

df = pd.read_parquet(weather_path)

df[df.duplicated("Time", keep=False)].sort_values("Time")

,Time,AirTemp,Humidity,Pressure,Rainfall,TrackTemp,WindDirection,WindSpeed
101,0 days 01:41:19.753000,19.1,69.0,939.9,False,23.2,60,1.2
353,0 days 01:41:19.753000,17.7,74.1,939.6,False,21.9,0,0.4
352,0 days 01:41:19.753000,17.6,74.1,939.5,False,21.7,0,0.0
351,0 days 01:41:19.753000,17.5,73.6,939.5,False,21.7,101,0.0
350,0 days 01:41:19.753000,17.5,73.8,939.6,False,21.7,164,0.0
...,...,...,...,...,...,...,...,...
220,0 days 01:41:19.753000,17.6,74.1,939.3,False,23.2,264,0.0
219,0 days 01:41:19.753000,17.7,73.1,939.2,False,23.4,196,0.7
218,0 days 01:41:19.753000,17.7,74.3,939.2,False,23.4,120,0.6
227,0 days 01:41:19.753000,18.0,73.2,939.2,False,22.6,311,0.4


In [34]:
df.shape

(472, 8)

In [35]:
df["Time"].nunique()

102

In [36]:
df["Time"].value_counts().head(10)

Time
0 days 01:41:19.753000    371
0 days 00:00:19.429000      1
0 days 00:02:19.443000      1
0 days 00:03:19.449000      1
0 days 00:04:19.436000      1
0 days 00:01:19.438000      1
0 days 00:06:19.433000      1
0 days 00:07:19.444000      1
0 days 00:08:19.452000      1
0 days 00:09:19.437000      1
Name: count, dtype: int64

In [37]:
df.tail(20)

,Time,AirTemp,Humidity,Pressure,Rainfall,TrackTemp,WindDirection,WindSpeed
452,0 days 01:41:19.753000,17.5,73.8,939.6,False,21.7,164,0.0
453,0 days 01:41:19.753000,17.5,73.6,939.5,False,21.7,101,0.0
454,0 days 01:41:19.753000,17.6,74.1,939.5,False,21.7,0,0.0
455,0 days 01:41:19.753000,17.7,74.1,939.6,False,21.9,0,0.4
456,0 days 01:41:19.753000,17.9,74.0,939.6,False,22.0,0,0.4
457,0 days 01:41:19.753000,18.0,73.8,939.6,False,21.9,0,0.0
458,0 days 01:41:19.753000,18.2,72.6,939.5,True,21.9,356,0.4
459,0 days 01:41:19.753000,18.3,72.8,939.6,True,22.0,115,0.6
460,0 days 01:41:19.753000,18.4,72.1,939.5,False,21.9,85,1.5
461,0 days 01:41:19.753000,18.4,71.2,939.5,False,21.9,81,1.6


In [38]:
duplicate_block = df[df["Time"] == df["Time"].value_counts().idxmax()]

duplicate_block.describe(include="all")

,Time,AirTemp,Humidity,Pressure,Rainfall,TrackTemp,WindDirection,WindSpeed
count,371,371.000000,371.000000,371.000000,371,371.000000,371.000000,371.000000
unique,NaN,NaN,NaN,NaN,2,NaN,NaN,NaN
top,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN
freq,NaN,NaN,NaN,NaN,278,NaN,NaN,NaN
mean,0 days 01:41:19.753000,18.137466,72.509434,939.490027,NaN,22.521294,147.175202,0.910782
std,0 days 00:00:00,0.545372,1.765676,0.235344,NaN,0.600004,107.828187,0.600758
min,0 days 01:41:19.753000,17.300000,69.000000,939.100000,NaN,21.600000,0.000000,0.000000
25%,0 days 01:41:19.753000,17.600000,70.900000,939.300000,NaN,21.900000,79.000000,0.500000
50%,0 days 01:41:19.753000,18.100000,72.800000,939.500000,NaN,22.500000,96.000000,0.800000
75%,0 days 01:41:19.753000,18.600000,73.900000,939.600000,NaN,23.100000,256.000000,1.300000


NOTE : Weather timestamps are generally unique across the FastF1 archive.

A small number of sessions (~0.7%) contain duplicated timestamps, likely due to upstream data collection or export anomalies.

During integration, weather data will be preprocessed by aggregating duplicate timestamps into a single observation before temporal alignment with lap data.

-----------------------------------
## **Conclusions**

This notebook reverse-engineered the structure of the FastF1 archive to establish an evidence-based integration strategy for downstream machine learning.

## **Key Findings**

- Identified the five primary datasets:
  - Results
  - Laps
  - Weather
  - Car Telemetry
  - Position Data

- Confirmed that each dataset has a different analytical granularity.

- Validated candidate keys across the complete archive instead of relying on representative files.

- Detected a small number of weather timestamp anomalies (<1% of sessions), which will be handled during preprocessing.

- Determined that telemetry and position data should **not** be directly merged with lap data because they exist at a much finer temporal resolution.

- Established that telemetry and position datasets must first be aggregated into lap-level features before integration.

## **Final Integration Strategy**

The machine learning dataset will use:

> **One row = One Driver × One Lap**

The datasets will contribute information as follows:

| Dataset | Integration Strategy |
|----------|----------------------|
| Results | Direct merge using session and driver identifiers |
| Laps | Base fact table |
| Weather | Time-aligned to laps after preprocessing |
| Car Data | Aggregate telemetry features per lap |
| Position Data | Aggregate positional features per lap |

This design preserves the original information content while creating a machine learning–ready analytical dataset suitable for feature engineering and predictive modelling.

The implementation of this integration pipeline will be carried out in the next notebook.